In [4]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import sys
import re

# Import your model class (IMPORTANT: Make sure the path is correct)
from FFNN import DeepNN

def extract_info_from_path(path):
    """Extract parameters from path."""
    path_lower = path.lower()
    path_upper = path.upper()
    basename = os.path.basename(path)
    dist_type = None

    # Check for abbreviated identifiers
    if 'NE' in path_upper:
        dist_type = "NE"
    elif 'NL' in path_upper:
        dist_type = "NL"
    elif 'NP' in path_upper:
        dist_type = "NP"
    elif 'poly' in path_lower:
        dist_type = "NP"
    elif 'lin' in path_lower:
        dist_type = "NL"
    elif 'exp' in path_lower:
        dist_type = "NE"
    else:
        dist_type = "NP"  # Default to "NP"

    # Extract parameters using regex patterns
    input_dim_match = re.search(r'd(\d+)', basename)
    hidden_size_match = re.search(r'H(\d+)', basename)
    depth_match = re.search(r'D(\d+)', basename)
    alpha_match = re.search(r'a([\d\.]+)', basename)
    order_match = re.search(r'O[_]?(\d+)', basename)
    
    info = {"distribution_type": dist_type}
    
    if input_dim_match:
        info['input_dim'] = int(input_dim_match.group(1))
    if hidden_size_match:
        info['hidden_size'] = int(hidden_size_match.group(1))
    if depth_match:
        info['depth'] = int(depth_match.group(1))
    if alpha_match:
        info['alpha'] = float(alpha_match.group(1))
    if order_match:
        info['order_num'] = order_match.group(1)
    
    return info

def load_model_from_path(model_path, device):
    """Load a pretrained model from path to device."""
    try:
        print(f"Loading model from {model_path}")
        model_data = torch.load(model_path, map_location=device)
        
        # Handle different save formats
        if isinstance(model_data, dict) and 'model_state_dict' in model_data:
            model_state = model_data['model_state_dict']
            # Get model parameters
            input_dim = model_data.get('input_dim')
            hidden_size = model_data.get('hidden_size')
            depth = model_data.get('depth')
            mode = model_data.get('mode', 'standard')
            alignment = model_data.get('alignment', False)
            
            print(f"Found parameters in model dict: input_dim={input_dim}, hidden_size={hidden_size}, depth={depth}")
        else:
            # Assume it's just the state dict
            model_state = model_data
            # Extract parameters from filename
            filename = os.path.basename(model_path)
            params = extract_info_from_path(filename)
            input_dim = params.get('input_dim')
            hidden_size = params.get('hidden_size')
            depth = params.get('depth', 2)
            mode = 'standard'
            alignment = False
            
            print(f"Extracted parameters from filename: input_dim={input_dim}, hidden_size={hidden_size}, depth={depth}")
        
        # If parameters are missing, try to extract from directory name
        if input_dim is None or hidden_size is None or depth is None:
            dir_name = os.path.basename(os.path.dirname(model_path))
            params = extract_info_from_path(dir_name)
            
            if input_dim is None:
                input_dim = params.get('input_dim')
            if hidden_size is None:
                hidden_size = params.get('hidden_size')
            if depth is None:
                depth = params.get('depth', 2)
                
            print(f"Extracted parameters from directory: input_dim={input_dim}, hidden_size={hidden_size}, depth={depth}")
        
        # Check if we have all the required parameters
        if input_dim is None or hidden_size is None or depth is None:
            raise ValueError(f"Could not determine model parameters from {model_path}")
        
        # Create model instance
        model = DeepNN(
            input_dim, 
            hidden_size, 
            depth, 
            mode=mode, 
            alignment=alignment
        ).to(device)
        
        # Load state dict
        model.load_state_dict(model_state)
        model.eval()  # Set to evaluation mode
        
        print(f"Successfully loaded model with input_dim={input_dim}, hidden_size={hidden_size}, depth={depth}")
        
        # Extract alpha value from path for naming
        alpha = params.get('alpha', 1.0)
        dist_type = params.get('distribution_type', 'NP')
        order_num = params.get('order_num', '1')
        
        return model, input_dim, hidden_size, depth, alpha, dist_type, order_num
    
    except Exception as e:
        print(f"Error loading model from {model_path}: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None, None, None, None, None, None

def compute_last_hidden_kernel_unnormalized_spectrum(model, X):
    """
    Computes the eigenvalues of the unnormalized kernel (H^T H) of the penultimate layer.
    """
    with torch.no_grad():
        # Get all linear layers
        linear_layers = []
        for i, layer in enumerate(model.layers):
            if isinstance(layer, torch.nn.Linear):
                linear_layers.append(i)
        
        if len(linear_layers) < 2:
            print("WARNING: Not enough linear layers to compute penultimate features")
            return torch.zeros(1, device=X.device)
            
        # Get penultimate linear layer index
        penultimate_linear_idx = linear_layers[-2]
        
        # Process through layers up to the penultimate linear + activation
        features = X
        for i, layer in enumerate(model.layers):
            if i <= penultimate_linear_idx + 1:  # +1 to include the activation after the linear layer
                features = layer(features)
                
                # Print stats for the penultimate layer output
                if i == penultimate_linear_idx + 1:
                    has_nan = torch.isnan(features).any().item()
                    all_zeros = (features == 0).all().item()
                    min_val = features.min().item()
                    max_val = features.max().item()
                    mean_val = features.mean().item()
                    print(f"Penultimate features stats:")
                    print(f"  Shape: {features.shape}")
                    print(f"  Has NaN: {has_nan}")
                    print(f"  All zeros: {all_zeros}")
                    print(f"  Range: [{min_val}, {max_val}]")
                    print(f"  Mean: {mean_val}")
        
        # Compute kernel matrix
        K_unnorm = features.T @ features
        
        # Check kernel matrix
        has_nan_kernel = torch.isnan(K_unnorm).any().item()
        is_symmetric = torch.allclose(K_unnorm, K_unnorm.T, rtol=1e-5)
        diag_mean = torch.diagonal(K_unnorm).mean().item()
        print(f"Kernel matrix stats:")
        print(f"  Shape: {K_unnorm.shape}")
        print(f"  Has NaN: {has_nan_kernel}")
        print(f"  Is symmetric: {is_symmetric}")
        print(f"  Mean diagonal: {diag_mean}")
        
        # Compute eigenvalues
        try:
            eigenvalues = torch.linalg.eigvalsh(K_unnorm)
            
            # Check eigenvalues
            has_nan_eig = torch.isnan(eigenvalues).any().item()
            min_eig = eigenvalues.min().item()
            max_eig = eigenvalues.max().item()
            num_negative = (eigenvalues < 0).sum().item()
            print(f"Eigenvalue stats:")
            print(f"  Shape: {eigenvalues.shape}")
            print(f"  Has NaN: {has_nan_eig}")
            print(f"  Range: [{min_eig}, {max_eig}]")
            print(f"  Number of negative eigenvalues: {num_negative}")
            
            return eigenvalues
        except Exception as e:
            print(f"Error computing eigenvalues: {e}")
            return torch.zeros(K_unnorm.shape[0], device=X.device)

# Main execution
model_path = "/home/goring/TF_spectrum/pretrain/paper_grid_1503/PT_NP_d8_H64_D2_a4.0_O_2_20250315_232312/model_NP_d8_H64_D2_a4.0_O_2.pt"  # Change this to your model path
n_samples = 1000000  # Number of samples to generate
seed = 42  # Random seed
output_dir = "./spectrum_plots"  # Directory to save plots

# Create multiple sample sizes to test
sample_sizes = [1000, 10000, 100000, 1000000]

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Load model
model, input_dim, hidden_size, depth, alpha, dist_type, order_num = load_model_from_path(model_path, device)

if model is None:
    print("Failed to load model, exiting.")
    sys.exit(1)

# Set seed
torch.manual_seed(seed)
np.random.seed(seed)

# Get model info for plot
model_basename = os.path.basename(os.path.dirname(model_path))

# Create a figure for comparing all spectra
plt.figure(figsize=(12, 8))

# Test each sample size
for n_samples in sample_sizes:
    print(f"\nTesting with {n_samples} samples:")
    
    # Generate random data
    X = torch.randn(n_samples, input_dim, device=device)
    print(f"Generated random X tensor with shape {X.shape}")
    
    # Compute spectrum
    print("Computing kernel spectrum...")
    eigenvalues = compute_last_hidden_kernel_unnormalized_spectrum(model, X)
    
    # Sort in descending order for plotting
    spectrum = np.sort(eigenvalues.detach().cpu().numpy())[::-1]
    
    # Plot this spectrum
    plt.loglog(np.arange(1, len(spectrum) + 1), spectrum, '-o', markersize=3, 
               label=f'n_samples={n_samples}')
    
    # Print key stats
    print(f"Spectrum Statistics (n_samples={n_samples}):")
    print(f"  Maximum eigenvalue: {spectrum[0]:.4e}")
    print(f"  Minimum eigenvalue: {spectrum[-1]:.4e}")
    print(f"  Condition number: {spectrum[0]/max(spectrum[-1], 1e-12):.4e}")

# Finalize plot
plt.xlabel('Index')
plt.ylabel('Eigenvalue')
plt.title(f'Kernel Spectrum Comparison\n{dist_type}_d{input_dim}_H{hidden_size}_D{depth}_a{alpha}_O_{order_num}')
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.2)

# Save the comparison plot
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
plot_path = os.path.join(output_dir, f'spectrum_comparison_{model_basename}_{timestamp}.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"\nSaved spectrum comparison plot to {plot_path}")

Using device: cpu
Loading model from /home/goring/TF_spectrum/pretrain/paper_grid_1503/PT_NP_d8_H64_D2_a4.0_O_2_20250315_232312/model_NP_d8_H64_D2_a4.0_O_2.pt
Extracted parameters from filename: input_dim=8, hidden_size=64, depth=2
Successfully loaded model with input_dim=8, hidden_size=64, depth=2

Testing with 1000 samples:
Generated random X tensor with shape torch.Size([1000, 8])
Computing kernel spectrum...
Penultimate features stats:
  Shape: torch.Size([1000, 64])
  Has NaN: False
  All zeros: False
  Range: [0.0, 8.723193168640137]
  Mean: 0.4617929756641388
Kernel matrix stats:
  Shape: torch.Size([64, 64])
  Has NaN: False
  Is symmetric: True
  Mean diagonal: 929.287109375
Eigenvalue stats:
  Shape: torch.Size([64])
  Has NaN: False
  Range: [-8.189388722712465e-09, 55302.77734375]
  Number of negative eigenvalues: 1
Spectrum Statistics (n_samples=1000):
  Maximum eigenvalue: 5.5303e+04
  Minimum eigenvalue: -8.1894e-09
  Condition number: 5.5303e+16

Testing with 10000 samp

/tmp/ipykernel_488222/3151067848.py:61: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_data = torch.load(model_path, map_location=device)


Penultimate features stats:
  Shape: torch.Size([1000000, 64])
  Has NaN: False
  All zeros: False
  Range: [0.0, 18.189550399780273]
  Mean: 0.4788675010204315
Kernel matrix stats:
  Shape: torch.Size([64, 64])
  Has NaN: False
  Is symmetric: True
  Mean diagonal: 996104.4375
Eigenvalue stats:
  Shape: torch.Size([64])
  Has NaN: False
  Range: [2.1110658645629883, 58705516.0]
  Number of negative eigenvalues: 0
Spectrum Statistics (n_samples=1000000):
  Maximum eigenvalue: 5.8706e+07
  Minimum eigenvalue: 2.1111e+00
  Condition number: 2.7808e+07

Saved spectrum comparison plot to ./spectrum_plots/spectrum_comparison_PT_NP_d8_H64_D2_a4.0_O_2_20250315_232312_20250317_013810.png
